This notebook is used in tandem with "PHOEBEUnitTestingScript.py" to interpret the outputs of that script.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u

import itertools
import pickle

Here we configure the notebook to match the setup of the script. `'fn'` is the filename of the pickle file (with the file extension this time).

In [ ]:
fn = "simple_test.pkl"

with open(fn, 'rb') as f:
    loaded_dict = pickle.load(f)
Refl_ON = loaded_dict['Refl ON']
(eccs, incls, per0s, ps, qs) = loaded_dict['Orbital Parameters']

comb = itertools.product(eccs, incls, per0s, ps, qs)
comb_list = list(comb)
comb_cnt = len(comb_list)

print('eccs:', eccs)
print('incls:', incls)
print('per0s:', per0s)
print('qs:', qs)
print('ps:', ps)
print('Reflection ON:', Refl_ON)
print('\nSuccesful System Count:', len(loaded_dict) - 2)
print('Num Combinations:', comb_cnt)

Using the data from the run we calculate the amplitudes of each light curve and the Root Mean-Squared Difference (RMSD) between them

In [ ]:
periods = np.zeros(comb_cnt)
PHOEBE_A = np.zeros(comb_cnt)
eBEER_A = np.zeros(comb_cnt)
RMSDs = np.zeros(comb_cnt)

for i in range(comb_cnt):
    comb = comb_list[i]
    ecc, incl, per0, p, q = comb
    comb_str = f'lc_{p}_{int(ecc*10)}_{incl}_{per0}_{int(q*10)}'

    try:
        if not Refl_ON:
            times, t0, relative_flux, eBEER_flux = loaded_dict[comb_str]
        else:
            times, t0, relative_flux, fit_flux, rescaled = loaded_dict[comb_str]
    except:
        print(f'Could not find combination:', fr'e:{ecc}, incl:{incl}, omega:{per0}, Porb:{p}, q:{q}')
        continue

    if not Refl_ON: residuals = (relative_flux - eBEER_flux)
    else: residuals = (relative_flux - fit_flux)
    
    MSD = np.mean(residuals**2)
    RMSD = np.sqrt(MSD)
    RMSDs[i] = RMSD
    periods[i] = p

    if not Refl_ON:
        eBEER_A[i] = (np.max(eBEER_flux) - np.min(eBEER_flux)) / 2
    else:
        eBEER_A[i] = (np.max(fit_flux) - np.min(fit_flux)) / 2
    PHOEBE_A[i] = (np.max(relative_flux) - np.min(relative_flux)) / 2

In [ ]:
plt.scatter(periods, RMSDs)
plt.xscale('log')
plt.xlabel('Period (days)')
plt.ylabel('RMSD (ppm)')
plt.title('sys_config')
# plt.legend()

In [ ]:
plt.plot(periods, eBEER_A, 'b.', label= 'eBEER')
plt.plot(periods, PHOEBE_A, 'r.', label= 'PHOEBE')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Period (days)')
plt.ylabel('Amplitude (ppm)')
plt.title('LC Amplitude vs Orbital Period')
plt.legend()